# Decoding-Free Marginalization: one-click demo

Reproduction of *Decoding-Free Sampling Strategies for LLM Marginalization*
([arXiv:2510.20208](https://arxiv.org/abs/2510.20208)).

This notebook clones the repo, verifies the math, shows the marginal estimate
converging to the exact value, and runs the Q&A evaluation on a free GPU.

**Runtime:** set it to a GPU (Colab: Runtime > Change runtime type > T4).


## 1. Setup

In [ ]:
# Clone the repo (skip if already alongside this notebook).
import os
if not os.path.exists("lattice.py"):
    !git clone https://github.com/KarthikSubramanian07/decoding-free-marginalization.git
    %cd decoding-free-marginalization
!pip -q install -r requirements.txt

In [ ]:
# Gemma needs a one-time free license click on its Hugging Face model page,
# then a token from https://huggingface.co/settings/tokens (Read scope).
# Running this cell shows a login box; paste your token there.
from huggingface_hub import notebook_login
notebook_login()

## 2. Verify the math (no model needed)

The correctness gate: the lattice enumerates exactly and the estimator converges to the exact marginal.

In [ ]:
!pytest -q

## 3. Load a small model

Gemma-3-1B fits comfortably on a free T4. Swap in `meta-llama/Llama-2-7b-hf` with `load_in_4bit=True` as a heavier option.

In [ ]:
from scoring import load_model
# Base (-pt) model: marginalization reads raw next-token probabilities,
# so the pretrained base is the faithful choice over the instruction-tuned -it.
MODEL = "google/gemma-3-1b-pt"
model, tokenizer = load_model(MODEL, load_in_4bit=False)
print("loaded", MODEL)

## 4. Estimate a marginal, and watch it converge

The estimate is a lower bound that climbs toward the true marginal as more tokenizations are scored. On short strings we can draw the exact marginal (dashed) for comparison.

In [ ]:
from analysis import convergence_data, plot_convergence

strings = [" marginalization", " tokenizer", " probability"]
data = [convergence_data(model, tokenizer, s, k=256) for s in strings]
for d in data:
    print(f"{d['text']!r}: {d['n_tokenizations']} tokenizations, "
          f"{d['num_near_canonical']} near-canonical")
plot_convergence(data, out_path="results/convergence.png")
from IPython.display import Image
Image("results/convergence.png")

## 5. Q&A evaluation

Start small to keep it quick, then scale `--n-questions` to 250 for the full reproduction. Three methods per question: canonical-only, the decoding-free lattice estimator, and importance sampling.

In [ ]:
from experiments import load_qa_dataset, evaluate, print_table, write_summary

results = []
for name in ["arc", "openbookqa", "medmcqa"]:
    items = load_qa_dataset(name, n=50, seed=0)   # bump to 250 for the full run
    res = evaluate(model, tokenizer, name, items, k=64, n_is_samples=16, seed=0)
    results.append(res)

print_table(results)
write_summary(results)

## 6. Headline plots

In [ ]:
from analysis import plot_runtime, plot_underestimation
plot_runtime(results, out_path="results/runtime.png")
plot_underestimation(results, out_path="results/underestimation.png")
from IPython.display import Image, display
display(Image("results/runtime.png"))
display(Image("results/underestimation.png"))

## Notes

- The lattice method should be markedly faster than importance sampling (which
  runs one forward pass per generated token), and importance sampling should sit
  below the lattice estimate more than half the time.
- Accuracy, speedups, and the underestimation rate land in `results/`. Compare
  them against the paper's numbers in the README and write down where they match
  and where they diverge.
